# Models and retention budget optimisation

Segment customers, predict churn, estimate the value at risk, and choose which customers to target within a budget.

## 1. Imports and load the customer table



In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import kron, identity, csr_matrix
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                             precision_score, recall_score, silhouette_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

customers = pd.read_parquet("../data/processed/customer_snapshots.parquet")
train = customers[customers["snapshot"] == "train"].reset_index(drop=True)
test = customers[customers["snapshot"] == "test"].reset_index(drop=True)
score = customers[customers["snapshot"] == "score"].reset_index(drop=True)
print(len(train), "train |", len(test), "test |", len(score), "score customers")
results = {}

4295 train | 4952 test | 5852 score customers


/Users/aminarehan/Documents/Machine Learning ktp /customer-retention-engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Customer segmentation (k-means on RFM)



In [2]:
# Segment today's customers by Recency, Frequency and Monetary value (RFM).
# RFM values are heavily skewed, so we log-transform and standardise them first,
# otherwise k-means would be dominated by a few very large spenders.
rfm = score[["recency_days", "frequency", "monetary"]]
rfm_scaled = StandardScaler().fit_transform(np.log1p(rfm))

for k in range(2, 8):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(rfm_scaled)
    print(f"k={k}: silhouette {silhouette_score(rfm_scaled, labels):.3f}")

# k=2 scores highest but only splits "active" from "inactive". k=4 is the next best
# and gives more actionable groups, so we use 4 segments.
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(rfm_scaled)
score["segment_id"] = kmeans.labels_
profile = score.groupby("segment_id").agg(customers=("customer_id", "count"),
                                          recency_days=("recency_days", "median"),
                                          frequency=("frequency", "median"),
                                          monetary=("monetary", "median")).round(0)

# Names chosen by reading each segment's median profile.
SEGMENT_NAMES = {0: "Lapsed one-time buyers", 1: "Loyal high-value", 2: "Active developing", 3: "At-risk regulars"}
profile.index = profile.index.map(SEGMENT_NAMES)
score["segment"] = score["segment_id"].map(SEGMENT_NAMES)
results["segments"] = profile.reset_index().rename(columns={"segment_id": "segment"}).to_dict(orient="records")
profile

k=2: silhouette 0.436


k=3: silhouette 0.343


k=4: silhouette 0.358


k=5: silhouette 0.336


k=6: silhouette 0.327


k=7: silhouette 0.338


,customers,recency_days,frequency,monetary
segment_id,,,,
Lapsed one-time buyers,1982,400.0,1.0,261.0
Loyal high-value,1147,16.0,13.0,4889.0
Active developing,1236,23.0,3.0,706.0
At-risk regulars,1487,176.0,5.0,1430.0


## 3. Churn model: train/test split



In [3]:
FEATURES = ["recency_days", "tenure_days", "frequency", "monetary", "avg_order_value",
            "orders_last_90d", "spend_last_90d", "unique_products", "days_per_order",
            "cancelled_orders", "cancel_rate", "is_uk"]

# Time-based split: train on the earlier snapshot, test on the later one.
X_train, y_train = train[FEATURES], train["churned"]
X_test, y_test = test[FEATURES], test["churned"]
print("Train churn rate:", round(y_train.mean(), 3), "| Test churn rate:", round(y_test.mean(), 3))

Train churn rate: 0.541 | Test churn rate: 0.481


## 4. Churn model: baseline (logistic regression)



In [4]:
# Baseline model: logistic regression. Simple, interpretable, and a fair benchmark.
logreg = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                       LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)
p_logreg = logreg.predict_proba(X_test)[:, 1]
print("Logistic regression AUC:", round(roc_auc_score(y_test, p_logreg), 3))

Logistic regression AUC: 0.792


## 5. Churn model: gradient boosting



In [5]:
# Gradient boosting: combines many small decision trees, so it can capture
# non-linear patterns and interactions that logistic regression cannot.
gbm = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                                 subsample=0.8, random_state=42)
gbm.fit(X_train, y_train)
p_gbm = gbm.predict_proba(X_test)[:, 1]
print("Gradient boosting AUC:", round(roc_auc_score(y_test, p_gbm), 3))

Gradient boosting AUC: 0.78


## 6. Model evaluation and comparison



In [6]:
def evaluate(name, y_true, prob):
    pred = (prob >= 0.5).astype(int)
    return {"model": name, "AUC": roc_auc_score(y_true, prob),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred)}

# A simple business rule for comparison: "the longer since the last order, the more likely to churn".
recency_rule = X_test["recency_days"] / X_test["recency_days"].max()

comparison = pd.DataFrame([
    {"model": "Recency rule (no ML)", "AUC": roc_auc_score(y_test, recency_rule)},
    evaluate("Logistic regression", y_test, p_logreg),
    evaluate("Gradient boosting", y_test, p_gbm),
]).set_index("model").round(3)
display(comparison)
results["model_comparison"] = comparison.reset_index().to_dict(orient="records")

fig, ax = plt.subplots(figsize=(5, 4))
for name, prob in [("Recency rule", recency_rule), ("Logistic regression", p_logreg), ("Gradient boosting", p_gbm)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, prob):.2f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Churn model comparison (test period)"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("../reports/roc_curves.png", dpi=150)

,AUC,accuracy,precision,recall
model,,,,
Recency rule (no ML),0.770,NaN,NaN,NaN
Logistic regression,0.792,0.708,0.649,0.856
Gradient boosting,0.780,0.696,0.632,0.882


## 7. What drives churn (SHAP)



In [7]:
# Logistic regression scored the higher AUC, so it is our chosen model.
# SHAP explains how much each feature pushed each customer's prediction up or down.
transform = logreg[:-1]                       # the log + scaling steps
X_test_t = pd.DataFrame(transform.transform(X_test), columns=FEATURES)
explainer = shap.LinearExplainer(logreg[-1], pd.DataFrame(transform.transform(X_train), columns=FEATURES))
shap_values = explainer.shap_values(X_test_t)

importance = (pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURES)
                .sort_values(ascending=False).round(3))
display(importance.to_frame("mean |SHAP|"))
results["top_churn_drivers"] = importance.head(5).index.tolist()

plt.figure()
shap.summary_plot(shap_values, X_test_t, show=False, max_display=10)
plt.tight_layout(); plt.savefig("../reports/shap_summary.png", dpi=150, bbox_inches="tight")

Background dataset has 4295 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=4295 when initializing the masker.


,mean |SHAP|
frequency,0.548
days_per_order,0.351
tenure_days,0.310
cancelled_orders,0.180
orders_last_90d,0.176
recency_days,0.164
avg_order_value,0.163
is_uk,0.080
spend_last_90d,0.070
monetary,0.068


## 8. Customer value at risk



In [8]:
# 1) Retrain the chosen model on all labelled data (train + test) before scoring today's customers.
labelled = pd.concat([train, test], ignore_index=True)
final_model = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                            LogisticRegression(max_iter=1000))
final_model.fit(labelled[FEATURES], labelled["churned"])
score["churn_prob"] = final_model.predict_proba(score[FEATURES])[:, 1]

# 2) Estimate each customer's 6-month value if they stay: their historical spend run-rate.
#    (Tenure is floored at 90 days so very new customers are not over-valued.)
def six_month_value(df):
    return df["monetary"] / df["tenure_days"].clip(lower=90) * 180

# Check the estimate against what customers who stayed actually spent in the test period.
stayed = test[test["churned"] == 0]
est, actual = six_month_value(stayed), stayed["future_spend"]
print("Value estimate vs actual spend (customers who stayed):")
print("  rank correlation:", round(est.corr(actual, method="spearman"), 3))
print("  total estimated £{:,.0f} vs actual £{:,.0f}".format(est.sum(), actual.sum()))

# 3) Value at risk = probability of churning x value if retained.
score["value_6m"] = six_month_value(score)
score["value_at_risk"] = score["churn_prob"] * score["value_6m"]
print()
print("Total 6-month value at risk across current customers: £{:,.0f}".format(score["value_at_risk"].sum()))
results["value_estimate_rank_corr"] = round(est.corr(actual, method="spearman"), 3)
results["total_value_at_risk"] = round(score["value_at_risk"].sum())
score.sort_values("value_at_risk", ascending=False)[["customer_id", "churn_prob", "value_6m", "value_at_risk"]].head(10)

Value estimate vs actual spend (customers who stayed):
  rank correlation: 0.669
  total estimated £3,901,194 vs actual £4,188,527

Total 6-month value at risk across current customers: £1,226,104


,customer_id,churn_prob,value_6m,value_at_risk
3351,15749,0.465657,11676.090361,5437.053501
3596,16000,0.208027,24787.400000,5156.460390
397,12752,0.512393,8733.560000,4475.016673
335,12688,0.547954,7763.591150,4254.089151
239,12590,0.509476,7959.651185,4055.252137
2801,15195,0.429693,7722.000000,3318.092781
1314,13687,0.675320,4882.536986,3297.274448
1053,13422,0.512570,5203.100000,2666.952354
5703,18139,0.151146,16786.440000,2537.208048
5616,18052,0.712857,3471.440426,2474.639753


## 9. Retention budget optimisation (linear programming)



In [9]:
# Which customers should receive which retention offer, within a fixed budget?
# ILLUSTRATIVE ASSUMPTIONS (a real project would measure these with A/B tests):
BUDGET = 5000          # £ available this period
CALL_CAPACITY = 300    # customers the team can call
OFFERS = {             # cost per customer, share of would-be churners it retains
    "email_voucher": {"cost": 2.0,  "save_rate": 0.05},
    "call_and_voucher": {"cost": 25.0, "save_rate": 0.25},
}

n, names = len(score), list(OFFERS)
cost = np.array([OFFERS[o]["cost"] for o in names])
save = np.array([OFFERS[o]["save_rate"] for o in names])
gain = score["value_at_risk"].to_numpy()[:, None] * save[None, :]      # expected revenue saved
net = (gain - cost[None, :]).ravel()                                    # one variable per customer x offer

within_budget = LinearConstraint(np.tile(cost, n)[None, :], 0, BUDGET)
call_limit = LinearConstraint(np.tile([0, 1], n)[None, :], 0, CALL_CAPACITY)

# Each customer gets at most one offer (sparse matrix keeps this fast for 5,000+ customers).
one_offer_each = LinearConstraint(kron(identity(n), csr_matrix(np.ones((1, len(names))))), 0, 1)

solution = milp(c=-net, constraints=[one_offer_each, within_budget, call_limit],
                integrality=np.ones(n * len(names)), bounds=Bounds(0, 1))
x = solution.x.reshape(n, len(names)).round().astype(int)
print(solution.message)

def summarise(choice, label):
    spent = (choice * cost).sum()
    saved = (choice * gain).sum()
    return {"strategy": label, "customers targeted": int(choice.sum()),
            "calls": int(choice[:, 1].sum()), "spend £": round(spent),
            "expected revenue saved £": round(saved), "net benefit £": round(saved - spent)}

# Comparison: a common rule of thumb - call the customers most likely to churn,
# then email the next ones, with the same budget.
naive = np.zeros_like(x)
order = score["churn_prob"].sort_values(ascending=False).index.to_numpy()
calls = int(min(CALL_CAPACITY, BUDGET // cost[1]))
emails = int((BUDGET - calls * cost[1]) // cost[0])
naive[order[:calls], 1] = 1
naive[order[calls:calls + emails], 0] = 1

strategies = pd.DataFrame([summarise(naive, "Rule of thumb: highest churn risk first"),
                           summarise(x, "Optimised (value at risk + budget)")]).set_index("strategy")
display(strategies)
results["strategies"] = strategies.reset_index().to_dict(orient="records")
results["assumptions"] = {"budget": BUDGET, "call_capacity": CALL_CAPACITY, "offers": OFFERS}

Optimization terminated successfully. (HiGHS Status 7: Optimal)


,customers targeted,calls,spend £,expected revenue saved £,net benefit £
strategy,,,,,
Rule of thumb: highest churn risk first,200,200,5000,1069,-3931
Optimised (value at risk + budget),1120,120,5000,64994,59994


## 10. Results and business recommendations



In [10]:
score["offer"] = np.where(x[:, 1] == 1, "call_and_voucher", np.where(x[:, 0] == 1, "email_voucher", "none"))
targets = score[score["offer"] != "none"].sort_values("value_at_risk", ascending=False)
targets[["customer_id", "churn_prob", "value_6m", "value_at_risk", "offer"]].to_csv(
    "../reports/recommended_retention_targets.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 3))
strategies["net benefit £"].plot.barh(ax=ax, color=["#9AA9AE", "#0E6E6A"])
ax.set_xlabel("Expected net benefit (£)"); ax.set_ylabel("")
ax.set_title(f"Same £{BUDGET:,} budget, different targeting")
fig.tight_layout(); fig.savefig("../reports/strategy_comparison.png", dpi=150)

with open("../reports/results.json", "w") as f:
    json.dump(results, f, indent=2, default=float)

print("Targets saved:", len(targets), "customers")
print(json.dumps(results, indent=2, default=float))

Targets saved: 1120 customers
{
  "segments": [
    {
      "segment": "Lapsed one-time buyers",
      "customers": 1982,
      "recency_days": 400.0,
      "frequency": 1.0,
      "monetary": 261.0
    },
    {
      "segment": "Loyal high-value",
      "customers": 1147,
      "recency_days": 16.0,
      "frequency": 13.0,
      "monetary": 4889.0
    },
    {
      "segment": "Active developing",
      "customers": 1236,
      "recency_days": 23.0,
      "frequency": 3.0,
      "monetary": 706.0
    },
    {
      "segment": "At-risk regulars",
      "customers": 1487,
      "recency_days": 176.0,
      "frequency": 5.0,
      "monetary": 1430.0
    }
  ],
  "model_comparison": [
    {
      "model": "Recency rule (no ML)",
      "AUC": 0.77,
      "accuracy": NaN,
      "precision": NaN,
      "recall": NaN
    },
    {
      "model": "Logistic regression",
      "AUC": 0.792,
      "accuracy": 0.708,
      "precision": 0.649,
      "recall": 0.856
    },
    {
      "model": "Grad